In [ ]:
import pandas as pd
df = pd.read_csv("fire_image_captions_v2.csv")
df

In [ ]:
import re

# Extract the number from the image_path (e.g., image123.jpg → 123)
df['sort_key'] = df['image_path'].apply(lambda x: int(re.search(r'image(\d+)', x).group(1)))

# Sort by the extracted number
df_sorted = df.sort_values(by='sort_key')

# Drop the helper column
df_sorted = df_sorted.drop(columns=['sort_key'])

# Save the sorted CSV
df_sorted.to_csv('fire_image_captions_sorted.csv', index=False)

print("Numerical sorting complete. Saved as 'fire_image_captions_sorted.csv'")


In [ ]:
import os
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

# Load the CSV
df = pd.read_csv("fire_image_captions.csv")

# Image and output directories
IMAGE_DIR = "fire_dataset_jpg"
OUTPUT_DIR = "fire_annotated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load font
try:
    font = ImageFont.truetype("arial.ttf", size=18)
except:
    font = ImageFont.load_default()

# Annotate images
for _, row in df.iterrows():
    image_path = row['image_path']
    caption = row['caption']
    label = row['label']

    try:
        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        # Compose text
        text = f"{label.upper()}"

        # Measure text size using textbbox
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        padding = 10
        new_height = img.height + text_height + 2 * padding

        # Create new image with extra space at bottom
        annotated_img = Image.new("RGB", (img.width, new_height), "white")
        annotated_img.paste(img, (0, 0))

        # Draw the text at the bottom
        draw = ImageDraw.Draw(annotated_img)
        draw.text((padding, img.height + padding), text, fill="black", font=font)

        # Save result
        fname = os.path.basename(image_path)
        annotated_img.save(os.path.join(OUTPUT_DIR, fname))
        print(f"✅ Annotated {fname}")
    except Exception as e:
        print(f"❌ Failed to annotate {image_path}: {e}")
